In [1]:
import os
import numpy as np
import pandas as pd
import dabest
from datetime import datetime
from dabest._stats_tools.confint_1group import summary_ci_1group

import warnings
warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

date = datetime.today().strftime('%Y%m%d')

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 43.93it/s]

Numba compilation complete!


In [4]:
homecomp = "D:\\"
labcomp = "C:\\Users\\User\\"

specifiedpath = labcomp
input_dir = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Worm tracking\\ML\\Processed\\Reprocced\\"
output_dir = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Worm tracking\\ML\\Processed\\Reprocced\\"
os.makedirs(output_dir, exist_ok=True)

In [5]:
def totalplot(tpath1, genotype, conc):
    """Process speed data for a genotype at a given ATR concentration."""
    df = pd.read_csv(tpath1 + genotype + ".csv")
    df = df.drop(columns=df.columns[0], axis=1)
    
    rr = df.copy()
    rr['Timer'] = 60
    rr.loc[df['second'] < 50.0, 'Timer'] = 50
    rr.loc[df['second'] < 40.0, 'Timer'] = 40
    rr.loc[df['second'] < 30.0, 'Timer'] = 30
    rr.loc[df['second'] < 20.0, 'Timer'] = 20
    rr.loc[df['second'] < 10.0, 'Timer'] = 10
    
    dff2 = pd.DataFrame()
    dff2[' Timer'] = rr['Timer']
    word = str(conc) + "x_.*"
    
    test2 = rr.filter(regex=genotype + ".*")
    dff2 = pd.concat([dff2, test2.filter(regex=word)], axis=1)
    
    trp = pd.DataFrame()
    for f in range(10, 70, 10):
        sp = dff2.iloc[:, 1:][(dff2[' Timer'] == f)].mean(axis=0).reset_index()
        sp.rename(columns={sp.columns[1]: "metric"}, inplace=True)
        sp["category"] = genotype + " " + str(conc) + "x ATR_" + str(f)
        sp['timer'] = str(f)
        sp['conc'] = str(conc)
        sp['genre'] = genotype
        trp = pd.concat([trp, sp], axis=0)
    
    return trp.reset_index(drop=True)

In [6]:
def calculate_stats(input_dir, responder, min_samples=2):
    """Calculate stats comparing 0x vs other concentrations."""
    concentrations = [0, 0.5, 1, 2, 4]
    rows = []
    
    if responder == "WT-Green":
        genotype_str = "w1118;snt1-P"
    else:
        genotype_str = f"snt1-P>{responder}"
    
    newtott = pd.DataFrame()
    for conc in concentrations:
        main = totalplot(input_dir, responder, conc)
        newtott = pd.concat([newtott, main], axis=0).reset_index(drop=True)
    wer = newtott[newtott['timer'] == '20']

    conc_data = {}
    for c in ['0x', '0.5x', '1x', '2x', '4x']:
        vals = wer[wer['category'] == responder + ' ' + c + ' ATR_20']['metric'].values
        conc_data[c] = vals[~np.isnan(vals)]
    
    # Skip if control (0x) has insufficient samples
    if len(conc_data['0x']) < min_samples:
        print(f"  Skipping {responder}: 0x has only {len(conc_data['0x'])} samples")
        return pd.DataFrame(rows)

    test_concs = ['0.5x', '1x', '2x', '4x']
    valid_test_concs = [c for c in test_concs if len(conc_data[c]) >= min_samples]
    
    if len(valid_test_concs) == 0:
        print(f"  Skipping {responder}: no test concentrations with >= {min_samples} samples")
        return pd.DataFrame(rows)
    
    idx_list = [responder + " 0x ATR_20"] + [responder + " " + c + " ATR_20" for c in valid_test_concs]
    idx_tuple = tuple(idx_list)
    
    valid_categories = [responder + " " + c + " ATR_20" for c in ['0x'] + valid_test_concs]
    wer_filtered = wer[wer['category'].isin(valid_categories)]
    
    p5 = dabest.load(wer_filtered, idx=idx_tuple, x="category", y="metric", id_col="index")
    results = p5.mean_diff.results
    
    for i, test_conc in enumerate(valid_test_concs):
        # Control row (0x)
        control_stats = summary_ci_1group(x=conc_data['0x'], func=np.mean, resamples=5000, alpha=95, random_seed=12345)
        rows.append({
            "Driver": "snt-1P",
            "Responder": responder,
            "Group": "Control",
            "Genotype": genotype_str,
            "ATR concentration": "0x",
            "Sample Size": len(conc_data['0x']),
            "Mean": round(control_stats['summary'], 4),
            "Mean_CI_Low": round(control_stats['bca_ci_low'], 4),
            "Mean_CI_High": round(control_stats['bca_ci_high'], 4),
            "Effect size": "",
            "Effect size_CI_Low": "",
            "Effect size_CI_High": "",
            "Delta Object": "",
            "Metric": "speed"
        })
        
        # Test row
        test_stats = summary_ci_1group(x=conc_data[test_conc], func=np.mean, resamples=5000, alpha=95, random_seed=12345)
        rows.append({
            "Driver": "snt-1P",
            "Responder": responder,
            "Group": "Test",
            "Genotype": genotype_str,
            "ATR concentration": test_conc,
            "Sample Size": len(conc_data[test_conc]),
            "Mean": round(test_stats['summary'], 4),
            "Mean_CI_Low": round(test_stats['bca_ci_low'], 4),
            "Mean_CI_High": round(test_stats['bca_ci_high'], 4),
            "Effect size": round(results.loc[i, 'difference'], 4),
            "Effect size_CI_Low": round(results.loc[i, 'bca_low'], 4),
            "Effect size_CI_High": round(results.loc[i, 'bca_high'], 4),
            "Delta Object": "Mean diff",
            "Metric": "speed"
        })
    
    return pd.DataFrame(rows)

In [7]:
responders = ["WT-Green", "ACR", "KCR1-ET", "KCR1GS", "KCR2"]
all_results = []

for responder in responders:
    print(f"Processing: {responder}")
    try:
        df_stats = calculate_stats(input_dir, responder)
        if len(df_stats) > 0:
            all_results.append(df_stats)
            df_stats.to_csv(output_dir + f"{responder}_thesis_stats.csv", index=False)
    except Exception as e:
        print(f"  Error: {e}")

Processing: WT-Green
Processing: ACR
Processing: KCR1-ET
Processing: KCR1GS
Processing: KCR2


In [8]:
df_all = pd.concat(all_results, ignore_index=True)
df_all.to_csv(output_dir + f"{date}_all_worm_thesis_stats.csv", index=False)
print(f"Saved combined file: {date}_all_worm_thesis_stats.csv")
df_all

Saved combined file: 20260207_all_worm_thesis_stats.csv


,Driver,Responder,Group,Genotype,ATR concentration,Sample Size,Mean,Mean_CI_Low,Mean_CI_High,Effect size,Effect size_CI_Low,Effect size_CI_High,Delta Object,Metric
0,snt-1P,WT-Green,Control,w1118;snt1-P,0x,21,-0.0551,-0.0662,-0.0452,,,,,speed
1,snt-1P,WT-Green,Test,w1118;snt1-P,0.5x,10,-0.0044,-0.0133,0.0076,0.0507,0.0368,0.067,Mean diff,speed
2,snt-1P,WT-Green,Control,w1118;snt1-P,0x,21,-0.0551,-0.0662,-0.0452,,,,,speed
3,snt-1P,WT-Green,Test,w1118;snt1-P,1x,10,-0.0588,-0.0720,-0.0430,-0.0037,-0.0209,0.0149,Mean diff,speed
4,snt-1P,WT-Green,Control,w1118;snt1-P,0x,21,-0.0551,-0.0662,-0.0452,,,,,speed
5,snt-1P,WT-Green,Test,w1118;snt1-P,2x,4,-0.0656,-0.0697,-0.0636,-0.0105,-0.0228,-0.0001,Mean diff,speed
6,snt-1P,WT-Green,Control,w1118;snt1-P,0x,21,-0.0551,-0.0662,-0.0452,,,,,speed
7,snt-1P,WT-Green,Test,w1118;snt1-P,4x,8,-0.0796,-0.0897,-0.0734,-0.0246,-0.0392,-0.0128,Mean diff,speed
8,snt-1P,ACR,Control,snt1-P>ACR,0x,3,-0.0281,-0.0323,-0.0212,,,,,speed
9,snt-1P,ACR,Test,snt1-P>ACR,0.5x,7,-0.1613,-0.2000,-0.1272,-0.1332,-0.1768,-0.1021,Mean diff,speed
